# NOAA coastal extreme water levels

Inspect NOAA CO-OPS annual high-water exceedance estimates for Sewells Point, Virginia (`8638610`). These are station-point historical estimates from NOAA's extreme-value analysis. The published `probability` is the annual exceedance probability (AEP); the 90% confidence bounds describe uncertainty in the estimated level, not additional probability knots. NOAA reports rounded probabilities and return periods separately, so this example retains both as published.

Levels use the source's reported datum and epoch. They are **not** asset flood depths or future sea-level projections; do not assume nearby locations share the same water level or datum. NOAA also supplies detrended annual extreme values. Its notice says those values are relative to a trendline and cannot simply be shifted to another datum. See [NOAA's product API documentation](https://api.tidesandcurrents.noaa.gov/dpapi/prod/). The [companion pipeline](../pipelines/noaa_extreme_water_levels_pipeline.py) saves the published values, annual series, context and raw-response checksums. Data source: NOAA National Ocean Service, CO-OPS.

Run from a local crc-docs checkout: use `uv sync`, then launch Jupyter with
`uv run jupyter lab` and select the project kernel. The first pipeline run
requires internet access.


In [1]:
# Check the local notebook environment before running the pipeline.
import importlib.util
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (root / "pipelines").is_dir():
    raise RuntimeError("Run this notebook from the crc-docs checkout or its notebooks directory.")
missing = [name for name in ("crc_sdk", "pyarrow", "pandas")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Run `uv sync` from the crc-docs checkout and select its kernel: {missing}")
sys.path.insert(0, str(root))
from pipelines.noaa_extreme_water_levels_pipeline import run
import pyarrow.parquet as pq

output = root / "pipeline_output" / "noaa_water_levels"
summary = run(output)
context = summary["context"]
print("Station:", context["station"]["stationName"])
print("Datum:", context["datum"], "epoch:", context["station"]["epoch"])
print("Trend:", context["trend"], context["trend_units"])
print("Published levels:", summary["levels"], "detrended annual values:", summary["annual_extremes"])
print("Annual series notice:", context["annual_notice"])

Station: Sewells Point
Datum: MHHW epoch: 1983-2001
Trend: 4.84 mm/yr
Published levels: 22 detrended annual values: 98
Annual series notice: The annual extreme values provided are after the monthly extreme series has been detrended. Annual values cannot be shifted to other datums as they aren't relative to a single datum level, but to a linear trendline through the mhhw or mllw datum at the mid-year of that datum's epoch.


In [2]:
levels = pq.read_table(output / "exceedance_levels.parquet").to_pandas()
levels[levels["published_return_period_years"].isin([2.0, 10.0, 50.0, 100.0])][[
    "annual_exceedance_probability", "published_return_period_years", "water_level_m",
    "lower_90ci_m", "upper_90ci_m", "datum", "datum_epoch"
]]

,annual_exceedance_probability,published_return_period_years,water_level_m,lower_90ci_m,upper_90ci_m,datum,datum_epoch
4,0.50,2.0,0.75,0.72,0.79,MHHW,1983-2001
11,0.10,10.0,1.12,1.04,1.23,MHHW,1983-2001
18,0.02,50.0,1.52,1.34,1.85,MHHW,1983-2001
21,0.01,100.0,1.71,1.47,2.19,MHHW,1983-2001


In [3]:
annual = pq.read_table(output / "annual_detrended.parquet").to_pandas()
print("Annual series:", annual["year"].min(), "to", annual["year"].max())
print("Resource snapshot checksums:")
for name, source in summary["sources"].items():
    print(name, source["sha256"], source["retrieved_at"])

Annual series: 1928 to 2025
Resource snapshot checksums:
station 248af4b034c113ec2a28e656bb5d9115d7af5424235263c18844acf921cca949 2026-10-30T19:43:19.054969+00:00
levels af8f1e7273070968c11cb46d292cfb465b3756cae9192c6ef2edaa1391254640 2026-10-30T19:43:19.336299+00:00
annuals c470cd959853cf881b3c9eb5789c96452ca184b323d09fd928d9339c2b8561c1 2026-10-30T19:43:19.539757+00:00


The three NOAA responses are fetched separately and can be revised; they are not an atomic upstream release. Keep all three cached responses and their checksums to replay this result. The adapter exposes NOAA's published estimates and does not refit a canonical hazard curve.